In [ ]:
!pip install skimpy

# Imports

In [ ]:
import numpy as np 
import pandas as pd 
import skimpy 
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier, XGBRFClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, RidgeClassifierCV
from sklearn.inspection import PartialDependenceDisplay
from scipy.stats import friedmanchisquare
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from colorama import Style, Fore
import matplotlib.pyplot as plt
from category_encoders import OneHotEncoder, TargetEncoder
import warnings
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
import seaborn as sns
import os
from itertools import combinations
sns.set_theme(style = 'white', palette = 'colorblind')
pal = sns.color_palette('colorblind')

warnings.simplefilter(action='ignore', category=FutureWarning)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Data description

* **id**    : Unique ID for the customer
* **Gender:** Gender of the customer
* **Age**   : Age of the customer 
* **Driving_License:** 0 = Customer does not have DL, 1 = Customer already has DL
* **Region_Code:** Unique code for the region of the customer
* **Previously_Insured:** 1 = Customer already has Vehicle Insurance, 0 = Customer doesn't have Vehicle Insurance
* **Vehicle_Age:** Age of the Vehicle
* **Vehicle_Damage:** 1= Customer got his/her vehicle damaged in the past. 0 = Customer didn't get his/her vehicle damaged in the past.
* **Annual_Premium:** The amount customer needs to pay as premium in the year
* **Policy_Sales_Channel:** Anonymized Code for the channel of outreaching to the customer ie. Different Agents, Over Mail, Over Phone, In Person, etc.
* **Vintage:** Number of Days, Customer has been associated with the company
* **Response:**  1=Customer is interested, 0=Customer is not interested


In [ ]:
# thanks for @ravi20076  https://www.kaggle.com/code/ravi20076/playgrounds4e07-autogluon-starter
def reduce_mem(df: pd.DataFrame):
    "This method reduces memory for numeric columns in the dataframe";

    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64', "uint16", "uint32", "uint64"];
    start_mem = df.memory_usage().sum() / 1024**2;

    for col in df.columns:
        col_type = df[col].dtypes

        if col_type in numerics:
            c_min = df[col].min();
            c_max = df[col].max();

            if "int" in str(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min >= np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min >= np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                if c_min >= np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)  

    end_mem = df.memory_usage().sum() / 1024**2

    print(f"Start - end memory:- {start_mem:5.2f} - {end_mem:5.2f} Mb");
    return df;


# Config

In [ ]:
SUBMIT = False
SEED = 42
N_SPLITS = 5

# Load data

In [ ]:
train=pd.read_csv('/kaggle/input/playground-series-s4e7/train.csv', index_col='id')
test=pd.read_csv('/kaggle/input/playground-series-s4e7/test.csv',index_col='id')

In [ ]:
train.head(3)

In [ ]:
test.head(3)

In [ ]:
print(f'{Style.BRIGHT}{Fore.YELLOW} SHAPE')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> train: {Fore.GREEN} {train.shape}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> test:  {Fore.GREEN} {test.shape}')

print(f'\n\n{Style.BRIGHT}{Fore.YELLOW} NULL VALUES')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.isnull().any().any()}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.isnull().any().any()}')

print(f'\n\n{Style.BRIGHT}{Fore.YELLOW} DUPLICATES')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.duplicated().any().any()}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.duplicated().any().any()}')


In [ ]:
cat_cols = ['Gender','Driving_License','Previously_Insured','Vehicle_Age','Vehicle_Damage']
num_cols = ['Age', 'Region_Code','Annual_Premium','Vintage']
target    = ['Response']
features = num_cols + cat_cols

for df in [train,test]:
    df[cat_cols] = df[cat_cols].astype('category')

In [ ]:
train = reduce_mem(train)
test = reduce_mem(test)

# Descriptive Statistics

In [ ]:
skimpy.skim(train)

* **Annual_Premium:** The Annual_Premium variable has a relatively high standard deviation (16450), which suggests a large dispersion of values in relation to the average (30460). This may indicate the presence of extreme values (outliers) in the data.

# EDA

In [ ]:
def plot_numerical():
    df = pd.concat([train[num_cols].assign(Source = 'Train'), 
                    test[num_cols].assign(Source = 'Test')], ignore_index = True)
    
    fig, axes = plt.subplots(len(num_cols), 3 ,figsize = (16, len(num_cols) * 4), 
                             gridspec_kw = {'hspace': 0.35, 'wspace': 0.3, 
                                            'width_ratios': [0.80, 0.20, 0.20]})

    for i,col in enumerate(num_cols):
        ax = axes[i,0]
        sns.kdeplot(data = df[[col, 'Source']], x = col, hue = 'Source', palette=['#456cf0', '#ed7647'], linewidth = 2.1, warn_singular=False, ax = ax) # Use of seaborn with artistic interface
        ax.set_title(f"\n{col}",fontsize = 9)
        ax.grid(visible=True, which = 'both', linestyle = '--', color='lightgrey', linewidth = 0.75)
        ax.set(xlabel = '', ylabel = '')

        ax = axes[i,1]
        sns.boxplot(data = df.loc[df.Source == 'Train', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#456cf0', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Train", fontsize = 9)

        ax = axes[i,2]
        sns.boxplot(data = df.loc[df.Source == 'Test', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#ed7647', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Test", fontsize = 9)

    plt.suptitle(f'\nDistribution analysis - numerical features',fontsize = 12, y = 0.95, x = 0.57, fontweight='bold')
    plt.show()



In [ ]:
plot_numerical()

In [ ]:
def plot_cat(limit_unique=20):
    selectcols = train[cat_cols].nunique()<=limit_unique
    cols_ = selectcols[selectcols].index.to_list()
    n_cols = len(cols_)
    fig, ax = plt.subplots(n_cols, 2, figsize=(12, 4 * n_cols))
    for i, coluna in enumerate(cols_):    
        sns.countplot(x=train[coluna], ax=ax[i, 0])
        ax[i, 0].set_title(f'{coluna}')
        ax[i, 0].set_ylabel('Count')
        ax[i, 0].set_xlabel(coluna)
        ax[i, 0].tick_params(axis='x', labelrotation=45)

        for container in ax[i, 0].containers:
            ax[i, 0].bar_label(container, fmt='%d', label_type='center', rotation=90)
            

        s1 = train[coluna].value_counts()        

        textprops = {
            'size':8, 
            'weight': 'bold', 
            'color':'white'
        }

        ax[i, 1].pie(s1,
            autopct='%2.2f%%',
            pctdistance=0.8, 
            textprops=textprops,
            labels=train[coluna].value_counts().index
        )    
        ax[i, 1].set_title(f'% {coluna}')

    plt.tight_layout()
    plt.show()

In [ ]:
%%time
plot_cat()

## Target

In [ ]:
ax = train[target].value_counts().plot(kind='bar')
colors = ['blue', 'green']
value_counts = train[target].value_counts()
percentages = (value_counts / value_counts.sum()) * 100
ax = value_counts.plot(kind='bar', figsize=(7,6))
for i, v in enumerate(value_counts):
    ax.text(i, v+0.5, f'{percentages[i]:.2f}%', ha='center', va='bottom')
plt.title('target');

* we have a problem with unbalanced classes.

# Cross validate

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS,shuffle=True, random_state=SEED)
oof, score, test_preds = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

In [ ]:
def crossvalidate(estimator, label = ''):
    
    X = train.copy()
    y = X.pop('Response')
    
    oof_ = np.zeros((len(X)))
    y_pred_test = np.zeros((len(test)))
    train_scores, val_scores = [], []
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        model = clone(estimator)
        
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        
        train_preds = model.predict_proba(X_train)[:, 1]
        val_preds = model.predict_proba(X_val)[:, 1]
                  
        oof_[val_idx] += val_preds
                
        train_score = roc_auc_score(y_train, train_preds)
        val_score = roc_auc_score(y_val, val_preds)
        
        print(f'Fold {fold+1}: {val_score:.5f}')

        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'{Style.BRIGHT}{Fore.BLUE}Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}{Style.RESET_ALL}')
    
    if SUBMIT:
        X_train = train.copy()
        y_train = X_train.pop('Response')
        model = clone(estimator)

        model.fit(X_tr, y_tr)
        y_pred_test += model.predict(test)
    
    return val_scores, oof_, y_pred_test

# Models

In [ ]:
%%time
score['logisticRegression'], oof['logisticRegression'], test_preds['logisticRegression'] = crossvalidate(make_pipeline(OneHotEncoder(cat_cols),
                                                                                   StandardScaler(),
                                                                                   LogisticRegression(max_iter=1000)),'LogisticRegression')

In [ ]:
%%time
score['LGBM'], oof['LGBM'],test_preds['LGBM'] = crossvalidate(make_pipeline(TargetEncoder(cat_cols),LGBMClassifier(random_state=42,verbose=-1)),'LGBM')

In [ ]:
%%time
score['XGB'], oof['XGB'], test_preds['XGB'] = crossvalidate(make_pipeline(XGBClassifier(random_state=42,verbosity=0,enable_categorical=True)),'XGB')

* **XGBRFClassifier**: Implements the Random Forest algorithm, where several models (decision trees) are trained independently and their variations are combined.

In [ ]:
%%time
score['RF'], oof['RF'], test_preds['RF'] = crossvalidate(make_pipeline(XGBRFClassifier(random_state=42,verbosity=0,enable_categorical=True)),'RF')

# Ensemble with Ridge


In [ ]:
walphas = np.arange(1,50,1)
ridgecv = RidgeClassifierCV(alphas=walphas)
ridgecv.fit(oof, train[target].values.ravel())

In [ ]:
w = RidgeClassifier(alpha=ridgecv.alpha_).fit(oof,train[target].values.ravel()).coef_[0]
w /= w.sum()
display(w)

In [ ]:
score['Ensemble_Ridge'] = roc_auc_score(train[target],oof.to_numpy() @ w)

# Combine models

In [ ]:
ensemble_scores = {}
models = oof.columns
for r in range(1,len(models)+1):
    for comb in combinations(models,r):
        combined_models = [label for label in comb]
        
        if len(combined_models)>1:
            oof_comb = np.mean(np.column_stack([oof[label] for label in combined_models]),axis=1)
            auc_mean = roc_auc_score(train[target],oof_comb)
        
            print(f'ensemble: {combined_models} - AUC: {auc_mean}')
            ensemble_scores['+'.join(comb)] = auc_mean


    ensemble_scores_sorted = sorted(ensemble_scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
ensemble_scores_sorted
for i, v in enumerate(ensemble_scores_sorted):
    score[v[0]] = v[1]

# Scores

In [ ]:
ax = score.mean().sort_values(ascending=True).plot(kind='barh', figsize=(8, 4*2), color='#a2a28f')

for container in ax.containers:
    ax.bar_label(container, label_type='center', color='black', fontsize=12, fontweight='bold')
ax.patches[-1].set_facecolor('#6ca957')

ax.set_title('Score Models', fontsize=16, fontweight='bold')

ax.set_xlabel('AUC', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')

ax.tick_params(axis='both', which='major', labelsize=12)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# Submission

In [ ]:

if SUBMIT:
    test_preds['Ensemble_Ridge'] = test_preds.to_numpy() @ w    
    sub = pd.read_csv('/kaggle/input/playground-series-s4e7/sample_submission.csv')
    sub[target] = test_preds['Ensemble_Ridge'] 
    sub.to_csv('submission.csv',index=False)
    